In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import os
import time
import random
import unicodedata
import pandas as pd


# ==================== 설정 ====================
BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
PLAYER_FILE = os.path.join(BASE_DIR, "players_by_season", "players_2026.xlsx")
PLAYER_FILE = unicodedata.normalize("NFC", PLAYER_FILE)

PERSON_COL = "person_no"   # 입력: person_no 로 사용할 열
BIRTH_COL  = "생년월일"     # 출력: YYYY.MM.DD (없으면 새로 생성)
ORIGIN_COL = "출신"         # 출력: 초 - 중 - 고 (없으면 새로 생성)

# 최종 실패(재시도 다 소진)한 person_no를 하나씩 즉시 기록하는 파일 — 중간에 끊겨도
# 이미 실패한 건 안 날아가게 매번 append로 열고 바로 닫는다(파일 없으면 자동 생성).
FAILED_FILE = os.path.join(os.path.dirname(PLAYER_FILE), "failed_person_no.txt")

VIEW_URL = "https://www.korea-baseball.com/info/player/player_view?person_no={}"

MIN_DELAY = 2.0
MAX_DELAY = 5.0
MAX_RETRY = 3
SAVE_EVERY = 50   # N명마다 중간 저장

BIRTH_RE = re.compile(r"(\d{4})\.(\d{2})\.(\d{2})")


# ==================== 파싱 ====================
def parse_player(person_no, session):
    """
    선수 프로필 페이지에서 (생년월일, 출신) 추출.
      - 반환 (birth, origin): 정상 응답. 못 찾으면 birth=None, origin='' 로 채워짐.
      - 반환 None: 요청 자체 실패 (재시도 대상).
    """
    url = VIEW_URL.format(person_no)

    try:
        res = session.get(url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{person_no}] 요청 실패: {e}")
        return None  # ← 재시도 대상

    res.encoding = "utf-8"

    try:
        soup = BeautifulSoup(res.text, "html.parser")

        # ── 생년월일: 요약 테이블 th=생년월일 → 옆 td 에서 YYYY.MM.DD 만 (나이 제외)
        birth = None
        summary = soup.select_one("div.summary_team")
        if summary:
            for tr in summary.select("tr"):
                th = tr.find("th")
                td = tr.find("td")
                if th and td and th.get_text(strip=True) == "생년월일":
                    m = BIRTH_RE.search(td.get_text(strip=True))
                    if m:
                        birth = f"{m.group(1)}.{m.group(2)}.{m.group(3)}"
                    break

        # ── 출신: 출신학교 섹션의 소속(span.title)을 맨 아래(오래된 초등) → 위 순으로
        origin = ""
        section = None
        for sec in soup.select("div.section_info"):
            h4 = sec.find("h4")
            if h4 and "출신학교" in h4.get_text():
                section = sec
                break

        if section:
            names = []  # DOM 순서(최신 → 오래됨)
            ul = section.select_one("div.profile_view ul")
            if ul:
                for li in ul.find_all("li", recursive=False):
                    title_span = li.find("span", class_="title")
                    if not title_span:
                        continue  # 헤더 행(지역/소속/연도/포지션) → title 없음 → 스킵
                    sort_span = li.find("span", class_="sort")
                    region = sort_span.get_text(strip=True) if sort_span else ""
                    name = title_span.get_text(strip=True)
                    # 지역 접두어 중복 제거 ("경기 수원북중" → "수원북중"), 없으면 그대로
                    if region and name.startswith(region + " "):
                        name = name[len(region) + 1:].strip()
                    if name:
                        names.append(name)

            # 아래(오래된 초등)부터 뒤집고 → 순서 유지 중복 제거
            seen = set()
            ordered = []
            for name in reversed(names):
                if name not in seen:
                    seen.add(name)
                    ordered.append(name)
            origin = " - ".join(ordered)

        return birth, origin

    except Exception as e:
        print(f"  [{person_no}] 파싱 오류: {e}")
        return None, ""  # 로드는 됐으나 파싱 실패 → 재시도 안 함


def _log_failed(person_no):
    """최종 실패한 person_no를 실패 로그 파일에 한 줄 즉시 append (파일 없으면 생성)."""
    with open(FAILED_FILE, "a", encoding="utf-8") as f:
        f.write(f"{person_no}\n")


# ==================== 메인 ====================
def crawl_player_profiles():
    if not os.path.exists(PLAYER_FILE):
        print(f"파일 없음: {PLAYER_FILE}")
        return

    df = pd.read_excel(PLAYER_FILE, engine="openpyxl")

    if PERSON_COL not in df.columns:
        print(f"'{PERSON_COL}' 열이 없습니다. 현재 열: {list(df.columns)}")
        return

    # 출력 열 없으면 생성 + object 타입으로 (문자열 할당 안전)
    for col in (BIRTH_COL, ORIGIN_COL):
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].astype(object)

    def is_filled(v):
        return pd.notna(v) and str(v).strip() != "" and str(v).strip().lower() != "nan"

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })
    # 세션 쿠키 워밍업 (목록 페이지 한 번 호출)
    try:
        session.get("https://www.korea-baseball.com/info/player/player_list", timeout=10)
    except Exception:
        pass

    # 대상: person_id 있고, 생년월일·출신이 둘 다 채워져 있지 않은 행 (재개 가능)
    targets = []
    for idx, row in df.iterrows():
        pno = row[PERSON_COL]
        if pd.isna(pno):
            continue
        pno = str(pno).strip().split(".")[0]  # 12345.0 → 12345 방지
        if not pno or pno.lower() == "nan":
            continue
        if is_filled(row[BIRTH_COL]) and is_filled(row[ORIGIN_COL]):
            continue  # 이미 둘 다 채워짐 → 스킵
        targets.append((idx, pno))

    total = len(targets)
    print(f"대상 선수: {total}명 (전체 {len(df)}행 중 / 이미 채워진 행 스킵)")

    failed = []

    for i, (idx, pno) in enumerate(targets, 1):
        result = None
        elapsed = 0.0

        for attempt in range(1, MAX_RETRY + 1):
            start = time.time()
            result = parse_player(pno, session)
            elapsed = time.time() - start

            if result is not None:
                break
            if attempt < MAX_RETRY:
                wait = random.uniform(MIN_DELAY, MAX_DELAY)
                print(f"  [{i}/{total}] {pno} → 실패 ({elapsed:.1f}s) / 재시도 {attempt}/{MAX_RETRY} ({wait:.1f}s 후)")
                time.sleep(wait)
            else:
                print(f"  [{i}/{total}] {pno} → 최종 실패 ({MAX_RETRY}회)")
                failed.append(pno)
                _log_failed(pno)

        if result is not None:
            birth, origin = result
            if birth:
                df.at[idx, BIRTH_COL] = birth
            if origin:
                df.at[idx, ORIGIN_COL] = origin
            print(f"  [{i}/{total}] {pno} → 생년월일={birth or '?'} / 출신={origin or '?'} ({elapsed:.1f}s)")

        # 중간 저장
        if i % SAVE_EVERY == 0:
            df.to_excel(PLAYER_FILE, index=False, engine="openpyxl")
            print(f"  >>> 중간 저장 완료 ({i}/{total})")

        # 9~11초 랜덤 딜레이 (응답시간 포함)
        target = random.uniform(MIN_DELAY, MAX_DELAY)
        wait = max(0, target - elapsed)
        time.sleep(wait)

    # 최종 저장
    df.to_excel(PLAYER_FILE, index=False, engine="openpyxl")

    print("\n" + "=" * 60)
    print(f"완료! 처리 {total}명 / 실패 {len(failed)}명")
    if failed:
        print(f"실패 person_id: {failed}")
        print(f"실패 목록 파일: {FAILED_FILE}")
    print("=" * 60)


# ==================== 실행 ====================
crawl_player_profiles()

In [ ]:
# ==================== 프로필 재크롤링 (등번호 · 포지션 · 투타) ====================
# 최초 수집 때 비어 있던 등번호/포지션/투타를, 지금은 프로필에 채워져 있으면 가져온다.
# (출신 크롤링과 같은 player_view 페이지의 요약 테이블 th='백넘버'/'포지션'/'투타'를 읽는다.)
# 세 필드 중 하나라도 비어 있는 선수만 대상 → 한 번의 페이지 요청으로 비어 있는 것만 채운다.
#
#   등번호: 유효번호(1~99)면 갱신 / 없으면 0 대신 100(=확인했으나 번호 없음)으로 표시.
#           고교야구는 세 자릿수 번호가 없으므로 100이 실제 번호와 안 겹친다. 세 자릿수
#           이상도 '번호 없음'으로 본다. (요청 실패면 0 유지 → 다음에 재시도)
#   포지션 : 페이지에 실제 값이 있으면 갱신 / 없으면(빈칸·'미지정') 마스터값 그대로.
#   투타   : 페이지에 실제 값이 있으면 갱신 / 없으면(빈칸·'미지정') 마스터값 그대로.
#           (포지션/투타는 등번호 100 같은 확정 마커가 없어, 채워질 때까지 매 실행 재조회됨)
import requests
from bs4 import BeautifulSoup
import os
import time
import random
import unicodedata
import pandas as pd


# ==================== 설정 ====================
BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
PLAYER_FILE = os.path.join(BASE_DIR, "players_by_season", "players_2026.xlsx")
PLAYER_FILE = unicodedata.normalize("NFC", PLAYER_FILE)

PERSON_COL = "person_no"   # 입력: person_no
BACKNUM_COL = "등번호"      # 출력: 백넘버 (0/빈칸인 행만 갱신)
POS_COL = "포지션"          # 출력: 포지션 (비어있는 행만 갱신)
BATTHROW_COL = "투타"       # 출력: 투타 (비어있는 행만 갱신)

NO_NUMBER_SENTINEL = 100   # 확인했으나 번호가 없는 선수에 찍는 값(고교는 세 자릿수 없음)

# 재크롤링에서 최종 실패한 person_no 기록 (출신 크롤링과 분리)
FAILED_FILE = os.path.join(os.path.dirname(PLAYER_FILE), "failed_profile_person_no.txt")

VIEW_URL = "https://www.korea-baseball.com/info/player/player_view?person_no={}"

MIN_DELAY = 2.0
MAX_DELAY = 5.0
MAX_RETRY = 3
SAVE_EVERY = 50   # N명마다 중간 저장


def _is_unassigned_backnum(v):
    """등번호가 '미지정'인지 — 빈칸/nan, 또는 0·00·000처럼 숫자로 0인 경우.
    (100 등 세 자릿수는 '확인했으나 없음' 마커라서 미지정으로 보지 않음 → 재크롤링 제외)"""
    if pd.isna(v):
        return True
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return True
    s = s.split(".")[0]  # 0.0 → 0
    return s.isdigit() and int(s) == 0


def _is_valid_backnum(s):
    """실제 등번호로 인정할 값 — 고교야구는 1~99만 유효(세 자릿수·0·빈칸 제외)."""
    return s.isdigit() and 1 <= int(s) <= 99


def _is_missing_text(v):
    """포지션/투타가 비어있는지 — 빈칸/nan, 또는 '미지정' 문자열."""
    if pd.isna(v):
        return True
    s = str(v).strip()
    return s == "" or s.lower() == "nan" or s == "미지정"


def _is_valid_text(s):
    """페이지에서 받은 포지션/투타를 실제 값으로 인정할지 — 빈칸·'미지정' 제외."""
    s = s.strip()
    return bool(s) and s != "미지정"


# ==================== 파싱 ====================
def parse_profile(person_no, session):
    """
    선수 프로필 요약 테이블에서 (백넘버, 포지션, 투타) 추출.
      - 반환 dict: 정상 응답. 각 값은 페이지 텍스트 그대로(없으면 '').
      - 반환 None: 요청 자체 실패 (재시도 대상).
    """
    url = VIEW_URL.format(person_no)

    try:
        res = session.get(url, timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{person_no}] 요청 실패: {e}")
        return None  # ← 재시도 대상

    res.encoding = "utf-8"

    try:
        fields = {"백넘버": "", "포지션": "", "투타": ""}
        soup = BeautifulSoup(res.text, "html.parser")
        summary = soup.select_one("div.summary_team")
        if summary:
            for tr in summary.select("tr"):
                th = tr.find("th")
                td = tr.find("td")
                if th and td:
                    key = th.get_text(strip=True)
                    if key in fields:
                        fields[key] = td.get_text(strip=True)
        return {"backnum": fields["백넘버"], "position": fields["포지션"], "batthrow": fields["투타"]}

    except Exception as e:
        print(f"  [{person_no}] 파싱 오류: {e}")
        return {"backnum": "", "position": "", "batthrow": ""}  # 로드는 됨 → 재시도 안 함


def _log_failed(person_no):
    """최종 실패한 person_no를 실패 로그 파일에 한 줄 즉시 append (파일 없으면 생성)."""
    with open(FAILED_FILE, "a", encoding="utf-8") as f:
        f.write(f"{person_no}\n")


# ==================== 메인 ====================
def crawl_profiles():
    if not os.path.exists(PLAYER_FILE):
        print(f"파일 없음: {PLAYER_FILE}")
        return

    df = pd.read_excel(PLAYER_FILE, engine="openpyxl")

    for col in (PERSON_COL, BACKNUM_COL, POS_COL, BATTHROW_COL):
        if col not in df.columns:
            print(f"'{col}' 열이 없습니다. 현재 열: {list(df.columns)}")
            return

    # 새 값을 안전하게 할당하기 위해 대상 열들을 object 타입으로
    for col in (BACKNUM_COL, POS_COL, BATTHROW_COL):
        df[col] = df[col].astype(object)

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })
    # 세션 쿠키 워밍업 (목록 페이지 한 번 호출)
    try:
        session.get("https://www.korea-baseball.com/info/player/player_list", timeout=10)
    except Exception:
        pass

    # 대상: person_no 있고, 등번호·포지션·투타 중 하나라도 비어 있는 행 (재개 가능)
    targets = []
    for idx, row in df.iterrows():
        pno = row[PERSON_COL]
        if pd.isna(pno):
            continue
        pno = str(pno).strip().split(".")[0]  # 12345.0 → 12345 방지
        if not pno or pno.lower() == "nan":
            continue
        need = (
            _is_unassigned_backnum(row[BACKNUM_COL])
            or _is_missing_text(row[POS_COL])
            or _is_missing_text(row[BATTHROW_COL])
        )
        if need:
            targets.append((idx, pno))

    total = len(targets)
    print(f"프로필 재크롤링 대상: {total}명 (전체 {len(df)}행 중 / 등번호·포지션·투타 중 결측 있는 행)")

    failed = []
    upd_no = upd_pos = upd_tb = 0   # 실제로 채운 필드별 카운트
    marked_none = 0                 # 등번호 100(번호 없음)으로 찍은 수

    for i, (idx, pno) in enumerate(targets, 1):
        result = None
        elapsed = 0.0

        for attempt in range(1, MAX_RETRY + 1):
            start = time.time()
            result = parse_profile(pno, session)
            elapsed = time.time() - start

            if result is not None:
                break
            if attempt < MAX_RETRY:
                wait = random.uniform(MIN_DELAY, MAX_DELAY)
                print(f"  [{i}/{total}] {pno} → 실패 ({elapsed:.1f}s) / 재시도 {attempt}/{MAX_RETRY} ({wait:.1f}s 후)")
                time.sleep(wait)
            else:
                print(f"  [{i}/{total}] {pno} → 최종 실패 ({MAX_RETRY}회)")
                failed.append(pno)
                _log_failed(pno)

        if result is not None:
            changes = []

            # ── 등번호: 현재 미지정(0/빈칸)인 행만
            if _is_unassigned_backnum(df.at[idx, BACKNUM_COL]):
                b = result["backnum"].strip()
                if _is_valid_backnum(b):
                    df.at[idx, BACKNUM_COL] = int(b)
                    upd_no += 1
                    changes.append(f"등번호={b}")
                else:
                    df.at[idx, BACKNUM_COL] = NO_NUMBER_SENTINEL
                    marked_none += 1
                    changes.append(f"등번호=없음({NO_NUMBER_SENTINEL})")

            # ── 포지션: 현재 비어있고(미지정/빈칸), 페이지에 실제 값이 있을 때만
            if _is_missing_text(df.at[idx, POS_COL]) and _is_valid_text(result["position"]):
                df.at[idx, POS_COL] = result["position"].strip()
                upd_pos += 1
                changes.append(f"포지션={result['position'].strip()}")

            # ── 투타: 현재 비어있고, 페이지에 실제 값이 있을 때만
            if _is_missing_text(df.at[idx, BATTHROW_COL]) and _is_valid_text(result["batthrow"]):
                df.at[idx, BATTHROW_COL] = result["batthrow"].strip()
                upd_tb += 1
                changes.append(f"투타={result['batthrow'].strip()}")

            msg = ", ".join(changes) if changes else "변경 없음"
            print(f"  [{i}/{total}] {pno} → {msg} ({elapsed:.1f}s)")

        # 중간 저장
        if i % SAVE_EVERY == 0:
            df.to_excel(PLAYER_FILE, index=False, engine="openpyxl")
            print(f"  >>> 중간 저장 완료 ({i}/{total}, 등번호 {upd_no}/없음 {marked_none} · 포지션 {upd_pos} · 투타 {upd_tb})")

        # 랜덤 딜레이 (응답시간 포함)
        target = random.uniform(MIN_DELAY, MAX_DELAY)
        wait = max(0, target - elapsed)
        time.sleep(wait)

    # 최종 저장
    df.to_excel(PLAYER_FILE, index=False, engine="openpyxl")

    print("\n" + "=" * 60)
    print(f"완료! 처리 {total}명")
    print(f"  등번호 갱신 {upd_no} / 번호없음(100) {marked_none} · 포지션 갱신 {upd_pos} · 투타 갱신 {upd_tb} · 실패 {len(failed)}")
    if failed:
        print(f"실패 person_no: {failed}")
        print(f"실패 목록 파일: {FAILED_FILE}")
    print("=" * 60)


# ==================== 실행 ====================
crawl_profiles()

In [ ]:
# ==================== 수비이닝 전체 초기 적재 (1단계) → player_defense JSON ====================
# gubun=P 페이지의 h4 "2026 시즌 | 우익수(1.2)" 에서 전체 선수의 포지션별 누적 수비이닝을 긁어
# player_defense_2026.json 에 person_no 키 하나로 저장한다(여러 포지션이어도 한 항목).
#
#   {
#     "201902005612": {"3루수": {"outs": 543, "innings": "181.0"},
#                      "유격수": {"outs": 30,  "innings": "10.0"},
#                      "2루수":  {"outs": 24,  "innings": "8.0"}},
#     "201902000459": {"우익수": {"outs": 5,   "innings": "1.2"}},
#     "201911111111": {}          ← 크롤은 했으나 수비기록 없음
#   }
#
#   ※ 이닝 표기는 야구식: "1.2" = 1이닝 2아웃(=5아웃), "181" = 181이닝.
#     outs(정수)가 실제 계산값(2단계 이닝 차 뺄셈용), innings 는 표시용.
#   ※ person_no 키가 있으면 '처리 완료' → 재개 시 스킵(별도 done 파일 불필요).
#   ※ players_2026.xlsx 는 person_no 목록으로만 읽고 절대 수정하지 않는다.
import requests
from bs4 import BeautifulSoup
import re
import os
import json
import time
import random
import unicodedata
import pandas as pd


# ==================== 설정 ====================
BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
PLAYER_FILE = unicodedata.normalize("NFC", os.path.join(BASE_DIR, "players_by_season", "players_2026.xlsx"))
DATA_DIR = os.path.dirname(PLAYER_FILE)
YEAR = 2026

CUM_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"player_defense_{YEAR}.json"))  # 누적(JSON)
FAILED_FILE = os.path.join(DATA_DIR, "failed_defense_person_no.txt")

PERSON_COL = "person_no"
VIEW_URL = "https://www.korea-baseball.com/info/player/player_view?person_no={}&gubun=P"  # gubun=P 필수
MIN_DELAY, MAX_DELAY, MAX_RETRY, SAVE_EVERY = 2.0, 5.0, 3, 50

# 수비 포지션만(투수·지명타자 제외 — 그들의 이닝은 수비이닝이 아님)
FIELD_POS = "포수|1루수|2루수|3루수|유격수|좌익수|중견수|우익수"
DEF_RE = re.compile(rf"({FIELD_POS})\(([\d.]+)\)")


# ==================== 이닝 <-> 아웃 (소수점=아웃, 0/1/2 뿐) ====================
def innings_to_outs(s):
    s = str(s).strip()
    if not s or s.lower() == "nan":
        return 0
    if "." in s:
        whole, frac = s.split(".", 1)
        return int(whole) * 3 + int(frac[:1] or 0)
    return int(s) * 3


def outs_to_innings(outs):
    outs = int(outs)
    return f"{outs // 3}.{outs % 3}"


# ==================== 파싱 ====================
def parse_defense(person_no, session):
    """gubun=P 프로필에서 그 시즌 포지션별 수비이닝 dict. 없으면 {}. 요청 실패면 None."""
    try:
        res = session.get(VIEW_URL.format(person_no), timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{person_no}] 요청 실패: {e}")
        return None
    res.encoding = "utf-8"
    try:
        soup = BeautifulSoup(res.text, "html.parser")
        result = {}
        for h4 in soup.find_all("h4"):
            text = h4.get_text(" ", strip=True)
            if str(YEAR) not in text:
                continue
            for pos, inn in DEF_RE.findall(text):
                result[pos] = inn
        return result
    except Exception as e:
        print(f"  [{person_no}] 파싱 오류: {e}")
        return {}


def _log_failed(pno):
    with open(FAILED_FILE, "a", encoding="utf-8") as f:
        f.write(f"{pno}\n")


# ==================== 메인 ====================
def build_player_defense():
    players = pd.read_excel(PLAYER_FILE, engine="openpyxl")
    if PERSON_COL not in players.columns:
        print(f"'{PERSON_COL}' 열이 없습니다.")
        return
    pnos = set()
    for x in players[PERSON_COL].dropna():
        p = str(x).strip().split(".")[0]
        if p.isdigit():
            pnos.add(int(p))
    pnos = sorted(pnos)

    # 기존 누적 로드 (키 존재 = 처리 완료 → 재개)
    cum = {}
    if os.path.exists(CUM_FILE):
        with open(CUM_FILE, encoding="utf-8") as f:
            cum = json.load(f)

    targets = [p for p in pnos if str(p) not in cum]
    print(f"수비이닝 초기 적재 대상 {len(targets)}명 (전체 {len(pnos)} / 완료 {len(cum)})")

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
    try:
        session.get("https://www.korea-baseball.com/info/player/player_list", timeout=10)
    except Exception:
        pass

    def _save():
        with open(CUM_FILE, "w", encoding="utf-8") as f:
            json.dump(cum, f, ensure_ascii=False, indent=1)

    failed = []
    with_def = 0
    for i, pno in enumerate(targets, 1):
        res = None
        elapsed = 0.0
        for attempt in range(1, MAX_RETRY + 1):
            start = time.time()
            res = parse_defense(pno, session)
            elapsed = time.time() - start
            if res is not None:
                break
            if attempt < MAX_RETRY:
                wait = random.uniform(MIN_DELAY, MAX_DELAY)
                print(f"  [{i}/{len(targets)}] {pno} → 실패 ({elapsed:.1f}s) / 재시도 {attempt}/{MAX_RETRY}")
                time.sleep(wait)
            else:
                print(f"  [{i}/{len(targets)}] {pno} → 최종 실패 ({MAX_RETRY}회)")
                failed.append(pno)
                _log_failed(pno)

        if res is not None:
            # 수비기록 없어도 {} 로 키를 남겨 '처리 완료' 표시(재개 시 스킵)
            cum[str(pno)] = {pos: {"outs": innings_to_outs(inn), "innings": inn} for pos, inn in res.items()}
            if res:
                with_def += 1
            summary = ", ".join(f"{p}({v})" for p, v in res.items()) if res else "수비기록 없음"
            print(f"  [{i}/{len(targets)}] {pno} → {summary} ({elapsed:.1f}s)")

        if i % SAVE_EVERY == 0:
            _save()
            print(f"  >>> 중간 저장 ({i}/{len(targets)}, 수비기록 {with_def})")

        target = random.uniform(MIN_DELAY, MAX_DELAY)
        time.sleep(max(0, target - elapsed))

    _save()
    print("\n" + "=" * 60)
    print(f"완료! 대상 {len(targets)} / 수비기록 있음 {with_def} / 실패 {len(failed)}")
    print(f"저장: {CUM_FILE} (총 {len(cum)}명)")
    if failed:
        print(f"실패 person_no: {failed} (→ {FAILED_FILE})")
    print("=" * 60)


# ==================== 실행 ====================
build_player_defense()

In [ ]:
# ==================== 수비이닝 누적 재구성 (재크롤 없음) → player_defense JSON ====================
# 이미 모아둔 수비이닝 데이터를 '다시 크롤하지 않고' 새 JSON 포맷으로 변환/재구성한다.
# 포맷을 바꿀 때마다 3~4시간짜리 전체 크롤을 다시 돌릴 필요가 없다.
#
#   소스 우선순위:
#     1) player_defense_2026.xlsx        (예전 롱포맷: person_no·position·outs·innings)
#     2) players_2026.xlsx 의 '수비이닝' 컬럼 (더 예전: 포지션별 JSON 문자열)
#   결과: player_defense_2026.json
#     {"201902005612": {"3루수": {"outs": 543, "innings": "181.0"}, ...}, ...}
#
#   ※ outs 를 기준값으로 삼아 innings 를 항상 'X.Y'(소수부=아웃 0/1/2)로 정규화한다.
#   ※ 기존 JSON이 있으면 지우지 않고 병합(변환분으로 덮어씀).
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


import os
import json
import unicodedata
import pandas as pd


# ==================== 설정 ====================
BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
PLAYER_FILE = unicodedata.normalize("NFC", os.path.join(BASE_DIR, "players_by_season", "players_2026.xlsx"))
DATA_DIR = os.path.dirname(PLAYER_FILE)
YEAR = 2026

CUM_JSON = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"player_defense_{YEAR}.json"))   # 결과
CUM_XLSX = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"player_defense_{YEAR}.xlsx"))   # 소스 1
DEF_COL = "수비이닝"                                                                              # 소스 2 (players 컬럼)


def innings_to_outs(s):
    s = str(s).strip()
    if not s or s.lower() == "nan":
        return 0
    if "." in s:
        whole, frac = s.split(".", 1)
        return int(whole) * 3 + int(frac[:1] or 0)
    return int(s) * 3


def outs_to_innings(outs):
    outs = int(outs)
    return f"{outs // 3}.{outs % 3}"


def _resolve(path):
    """Drive(NFC/NFD 혼재)에서 파일을 관대하게 찾는다. 못 찾으면 None."""
    if os.path.exists(path):
        return path
    d, base = os.path.dirname(path), os.path.basename(path)
    if not os.path.isdir(d):
        return None
    want = unicodedata.normalize('NFC', base)
    for f in os.listdir(d):
        if unicodedata.normalize('NFC', f) == want:
            return os.path.join(d, f)
    return None


def rebuild_player_defense():
    # 기존 JSON 있으면 불러와서 병합 (없어진 데이터가 생기지 않게)
    cum = {}
    _json_path = _resolve(CUM_JSON)
    if _json_path:
        with open(_json_path, encoding="utf-8") as f:
            cum = json.load(f)
        print(f"기존 JSON {len(cum)}명 로드 (병합)")

    converted = 0
    source = None

    # ── 소스 1: 롱포맷 xlsx ──
    _xlsx = _resolve(CUM_XLSX)
    if _xlsx:
        df = pd.read_excel(_xlsx, engine="openpyxl")
        need = {"person_no", "position"}
        if need.issubset(df.columns):
            for _, r in df.iterrows():
                if pd.isna(r.get("person_no")):
                    continue
                pos = str(r.get("position", "")).strip()
                if not pos or pos.lower() == "nan":
                    continue
                pno = str(int(float(r["person_no"])))
                if pd.notna(r.get("outs")):
                    outs = int(r["outs"])
                else:
                    outs = innings_to_outs(r.get("innings"))
                cum.setdefault(pno, {})[pos] = {"outs": outs, "innings": outs_to_innings(outs)}
                converted += 1
            source = os.path.basename(_xlsx)

    # ── 소스 2: players_2026 의 '수비이닝' JSON 컬럼 ──
    if source is None and os.path.exists(PLAYER_FILE):
        df = pd.read_excel(PLAYER_FILE, engine="openpyxl")
        if DEF_COL in df.columns and "person_no" in df.columns:
            for _, r in df.iterrows():
                if pd.isna(r.get("person_no")):
                    continue
                raw = r.get(DEF_COL)
                if pd.isna(raw) or str(raw).strip() in ("", "nan"):
                    continue
                try:
                    d = json.loads(str(raw))
                except Exception:
                    continue
                pno = str(int(float(r["person_no"])))
                entry = {}
                for pos, v in d.items():
                    outs = innings_to_outs(v)
                    entry[str(pos)] = {"outs": outs, "innings": outs_to_innings(outs)}
                cum[pno] = entry           # 빈 dict({})도 '처리 완료'로 유지
                converted += len(entry)
            source = f"{os.path.basename(PLAYER_FILE)} ('{DEF_COL}' 컬럼)"

    if source is None:
        print("변환할 소스가 없습니다.")
        print(f"  확인: {CUM_XLSX}")
        print(f"  확인: {PLAYER_FILE} 의 '{DEF_COL}' 컬럼")
        print(f"  [진단] DATA_DIR 존재: {os.path.isdir(DATA_DIR)}")
        if os.path.isdir(DATA_DIR):
            names = sorted(os.listdir(DATA_DIR))
            print(f"  [진단] 폴더 파일 {len(names)}개:")
            for n in names[:40]:
                print("      -", n)
        print("  → 파일이 안 보이면 Drive 마운트가 오래된 것: drive.mount('/content/drive', force_remount=True) 후 재실행")
        return

    with open(CUM_JSON, "w", encoding="utf-8") as f:
        json.dump(cum, f, ensure_ascii=False, indent=1)

    with_def = sum(1 for v in cum.values() if v)
    print("\n" + "=" * 60)
    print(f"재구성 완료! 소스: {source}")
    print(f"  선수 {len(cum)}명 (수비기록 있음 {with_def}) / 변환한 포지션 행 {converted}")
    print(f"  저장: {CUM_JSON}")
    print("=" * 60)

    # 샘플 3명 미리보기
    for i, (pno, d) in enumerate(cum.items()):
        if not d:
            continue
        print(f"  예시 {pno}: " + ", ".join(f"{p}({v['innings']}/{v['outs']}아웃)" for p, v in d.items()))
        if i >= 2:
            break


# ==================== 실행 ====================
rebuild_player_defense()